### Data Ingestion

In [2]:
from langchain_core.documents import Document

In [6]:
doc = Document(
    page_content = "This is a random document",
    metadata = {
        "source":"randomtxt.txt",
        "page":1
    }
)
doc

Document(metadata={'source': 'randomtxt.txt', 'page': 1}, page_content='This is a random document')

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../data/algo_book.pdf")

docs = list(loader.lazy_load())

1677


In [3]:
import re
from transformers import AutoTokenizer
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("../data/algo_book.pdf")
docs = list(loader.lazy_load())

def clean_clrs_page(text: str) -> str: # text is a single page in the book
    text = re.sub(r'(\n[A-Z\-]+\(.*?\)\n)', r'\n<<<ALGO_START>>>\1', text) # pattern example: HEAP-EXTRACT-MAX(A)
    text = re.sub(r'\n\d+\s+Chapter \d+.*?\n', '\n', text) # removes page headers like 123   Chapter 6 Heapsort
    text = re.sub(r'\nProblems for Chapter \d+.*', '', text, flags=re.DOTALL) # removes the exercise questions from the page
    return text

def remove_exercises(docs):
    theory_docs = []
    for doc in docs:
        content = doc.page_content
        exercise_split = re.split(r'\nExercises \d+\.\d+', content) # splits and returns an array where exercise_split[0] is the theory content and exercise_split[1] is the exercises content
        doc.page_content = clean_clrs_page(exercise_split[0]) # clean the theory content and update the document
        if len(doc.page_content.strip()) > 100: # only keep pages with substantial theory content (arbitrary threshold)
            theory_docs.append(doc)
    return theory_docs

clean_docs = remove_exercises(docs)
print(f"Pages after cleaning: {len(clean_docs)}")


Pages after cleaning: 1667


In [4]:
# ── OPEN SOURCE TOKEN LENGTH FUNCTION ─────────────────────────────────────
# BAAI/bge-base-en-v1.5 — best open source model for RAG embeddings
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5") #downloads and loads the tokenizer associated with this particular model

def token_length(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False)) # the encode function returns the token Ids List like [101, 102, 103, ...] that correspond to the and we take the length of that list to get the token count

# ── CHUNK ──────────────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,        # 400 tokens ≈ full theorem + proof in CLRS
    chunk_overlap=60,      # ~15% overlap to preserve reasoning context
    separators=[
        "\n\n\n",          # section breaks
        "\n\n",            # paragraph breaks
        "\n",              # line breaks (preserves pseudocode structure)
        ". ",              # sentence boundary
        " ",               # word boundary
        "",                # character fallback (last resort)
    ],
    length_function=token_length,
)

chunks = splitter.split_documents(clean_docs)
print(f"Total chunks: {len(chunks)}")

for i, chunk in enumerate(chunks[:5]):
    print(f"\nChunk {i}:")
    print(f"  Tokens : {token_length(chunk.page_content)}")
    print(f"  Page   : {chunk.metadata.get('page')}")
    print(f"  Preview: {chunk.page_content[:80].strip()}...")

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Total chunks: 2630

Chunk 0:
  Tokens : 44
  Page   : 2
  Preview: T homas H . Cormen
Charles E. Leiserson
Ronald L. Rivest
Clifford Stein
Introduc...

Chunk 1:
  Tokens : 329
  Page   : 3
  Preview: © 2022 M assachusetts Institute of Technology
All rights reserved. No part of th...

Chunk 2:
  Tokens : 153
  Page   : 4
  Preview: Contents
Copyright
Preface
I        Foundations
Introduction
1      The Role of...

Chunk 3:
  Tokens : 211
  Page   : 5
  Preview: ★     4.6      Proof of the continuous master theorem
 ★     4.7      Akra-Bazzi...

Chunk 4:
  Tokens : 179
  Page   : 6
  Preview: III     D ata Structures
Introduction
10    Elementary D ata Structures
10.1...
